In [6]:

import gzip
import json
import re
import random
from pathlib import Path

import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
metadata_path = RAW_DIR / "meta_Cell_Phones_and_Accessories.jsonl.gz"

# Confirm the dataset is available
print("Metadata file exists:", metadata_path.exists())
print("Metadata file:", metadata_path.name)


Metadata file exists: True
Metadata file: meta_Cell_Phones_and_Accessories.jsonl.gz


In [37]:

import re

EXCLUDE_TERMS = re.compile(
    r"\b("
    r"case|cover|screen protector|protector|charger|charging pad|"
    r"battery|headset|earbuds|stand|mount|holder|cable|adapter|"
    r"stylus|skin|replacement|digitizer|lcd assembly|screen assembly|"
    r"screen guard|tempered glass|wallet|kickstand|watch band|"
    r"strap|lanyard|dust plug|phone bag|cellphone bag|"
    r"signal booster|cellular repeater|charging dock|dock connector|"
    r"pop ?socket|pop ?grip|finger ring|armband|stabilizer|"
    r"phone cage|vr headset|google cardboard|sim card|"
    r"headphone bundle|headphone port|microphone replacement|"
    r"holster|belt clip|tripod|selfie stick|"
    r"ceramic film|screen guard|screen guards|"
    r"screen protector|screen protectors|"
    r"phone charm|smartphone charm|"
    r"tactical extreme|"
    r"piel frama|"
    r"game of thrones for apple iphone|"
    r"incharge auto|"
    r"phone fan|ring holder|replacement parts"
    r")\b",
    flags=re.IGNORECASE
)

PHONE_TERMS = re.compile(
    r"\b("
    r"smartphone|cell phone|mobile phone|"
    r"iphone|google pixel|pixel [0-9]|"
    r"samsung galaxy|galaxy [a-z]?\d|"
    r"motorola|moto [a-z0-9]+|"
    r"oneplus|xiaomi|redmi|poco|"
    r"nokia|oppo|vivo|realme|huawei|honor|"
    r"alcatel|sony xperia|tecno|infinix|"
    r"blu vivo|blu studio|cell allure|"
    r"unlocked phone|prepaid phone"
    r")\b",
    flags=re.IGNORECASE
)

def is_candidate_smartphone(title):
    if not isinstance(title, str) or not title.strip():
        return False

    if EXCLUDE_TERMS.search(title):
        return False

    return bool(PHONE_TERMS.search(title))

print("Updated smartphone filter loaded.")


Updated smartphone filter loaded.


In [38]:

random.seed(42)

sample_titles = []
titles_seen = 0
sample_size = 1000

with gzip.open(metadata_path, "rt", encoding="utf-8") as file:
    for line in file:
        product = json.loads(line)
        title = product.get("title")

        if not isinstance(title, str) or not title.strip():
            continue

        titles_seen += 1

        if len(sample_titles) < sample_size:
            sample_titles.append(title)
        else:
            j = random.randrange(titles_seen)
            if j < sample_size:
                sample_titles[j] = title

validation_df = pd.DataFrame({"title": sample_titles})
validation_df["candidate"] = validation_df["title"].apply(
    is_candidate_smartphone
)

predicted_phones = validation_df.loc[
    validation_df["candidate"], ["title"]
].reset_index(drop=True)

print("Predicted smartphone titles:", len(predicted_phones))
display(predicted_phones)


Predicted smartphone titles: 11


,title
0,"Apple iPhone 4S 8 GB AT&T, White"
1,"Samsung Galaxy S21+ 5G, US Version, 128GB, Phantom Violet - Unlocked (Renewed)"
2,Motorola RAZR V3 Unlocked Phone with Camera and Video Player-U.S. Version with Warranty (Pink)
3,"Samsung Galaxy A72 (A725M/DS) 128GB Dual SIM, GSM Unlocked, Android Smartphone (Awesome Black) (Renewed)"
4,TORU CX PRO iPhone 6 - Hot Pink
5,JANIZZ New Unlocked Simple Big Button Dual Screen Flip Mobile Cell Phone for Seniors Elderly with Sos Button and Cradle
6,"BLU Vivo XL5-6.3"" HD Display Smartphone, 64GB+ 3GB RAM- Black"
7,"Samsung Galaxy S6 Active G890A Unlocked GSM Smartphone, Blue"
8,"BLU Studio Mega (8GB) - 6.0"" HD Dual SIM GSM Factory Unlocked Smartphone (Black)"
9,"Motorola Moto One Macro (64GB, 4GB) 6.2”, Macro Vision Camera (5X Closer) Dual SIM GSM Unlocked (at&T/T-Mobile/MetroPCS/Cricket/H2O) XT2016-2 International Model (Space Blue, 64 GB)"


In [39]:

PHONE_HINTS = re.compile(
    r"\b(iphone|pixel|galaxy|oneplus|xiaomi|redmi|"
    r"motorola|moto|nokia|oppo|vivo|realme|huawei|"
    r"honor|alcatel|sony xperia|tecno|infinix|"
    r"smartphone|cell phone|mobile phone)\b",
    flags=re.IGNORECASE
)

ACCESSORY_HINTS = re.compile(
    r"\b("
    r"case|cover|screen protector|protector|charger|charging dock|"
    r"charging pad|battery|headset|earbuds|stand|mount|holder|"
    r"cable|adapter|stylus|skin|replacement|belt clip|holster|"
    r"screen guard|tempered glass|signal booster|cellular repeater|"
    r"cardboard|virtual reality|vr headset|armband|"
    r"phone stabilizer|stabilizer video rig|"
    r"watch|band|strap|lanyard|dust plug|phone bag|cellphone bag|"
    r"phone fan|pop socket|popsocket|ring holder|finger strap|"
    r"selfie stick|tripod|car mount|wallet|kickstand|"
    r"screen assembly|lcd screen|dock connector|"
    r"replacement screen|replacement belt clip|"
    r"phone charm|charm accessory|sim card|"
    r"headphone bundle|headphone port|microphone replacement"
    r")\b",
    flags=re.IGNORECASE
)

not_selected = validation_df.loc[
    ~validation_df["candidate"], "title"
].fillna("").astype(str)

potential_misses = not_selected[
    not_selected.apply(lambda title: bool(PHONE_HINTS.search(title)))
]

potential_misses = potential_misses[
    ~potential_misses.apply(
        lambda title: bool(ACCESSORY_HINTS.search(title))
    )
]

print("Potential missed products to inspect:", len(potential_misses))
display(potential_misses.to_frame(name="title").head(40))


Potential missed products to inspect: 7


,title
51,"Compatible with Anti-drop iPhone 11/XR, HD Ceramic Film, Anti-peep frosted Ceramic Film, Anti-fingerprint Ceramic Film, 2PCS"
492,Tech21 Evo Tactical Extreme Edition for iPhone 7 - Black
546,Love You To The Moon & Back Mom! Space Nebula Galaxy PopSockets PopGrip: Swappable Grip for Phones & Tablets
553,"eTECH Collection 5 Pack of Front & Back Combo PET Plastic Anti Glare Matted Screen Protectors for Apple iPhone 4S/4 AT&T, T-Mobile, Sprint, Verizon - from USA"
670,Piel Frama Unipur for Samsung Galaxy S4 (black)
706,XtremeMac IPU-IA2-13 InCharge Auto for iPhone/iPod/iPad
956,Popular Game of thrones for Apple Iphone 5 U115690


In [40]:
pd.set_option("display.max_colwidth", None)

display(
    potential_misses.reset_index(drop=True).to_frame(name="title")
)


,title
0,"Compatible with Anti-drop iPhone 11/XR, HD Ceramic Film, Anti-peep frosted Ceramic Film, Anti-fingerprint Ceramic Film, 2PCS"
1,Tech21 Evo Tactical Extreme Edition for iPhone 7 - Black
2,Love You To The Moon & Back Mom! Space Nebula Galaxy PopSockets PopGrip: Swappable Grip for Phones & Tablets
3,"eTECH Collection 5 Pack of Front & Back Combo PET Plastic Anti Glare Matted Screen Protectors for Apple iPhone 4S/4 AT&T, T-Mobile, Sprint, Verizon - from USA"
4,Piel Frama Unipur for Samsung Galaxy S4 (black)
5,XtremeMac IPU-IA2-13 InCharge Auto for iPhone/iPod/iPad
6,Popular Game of thrones for Apple Iphone 5 U115690


In [41]:
# Create a manual validation sample

validation_review = validation_df.copy()

validation_review["manual_label"] = ""

display(
    validation_review[["title", "candidate", "manual_label"]]
)

,title,candidate,manual_label
0,"LG K8V (VS500) Case, Capsule-Case Slim Fit Snap-on (Black) Hard Case for LG K8 V Verizon VS500 - (Red Panda)",False,
1,Adaptive Fast Wall Adapter Micro USB Charger for Samsung Galaxy S7 Edge S6 Note 5 4 J3 J5 J7 Prime Bundled with UrbanX Micro USB Cable Cord - 10ft and OTG Adapter - 3 Items - Fast Charging Kit - White,False,
2,Amzer AMZ93843 Silicone Jelly Skin Fit Case Cover for LG Optimus Elite LS969/VM969 - Skin - Retail Packaging - Black,False,
3,"Vinanker Case for Samsung Galaxy S9, Premium Leather Flip Wallet Cover with Card Slots Phone Case for Samsung Galaxy S9 (Red)",False,
4,"AGOZ Cell Phone CD Slot Car Mount Holder Cradle Compatible with iPhone 11 PRO XS MAX XR X 8 7, Galaxy S20, S10 5G, S10 Plus, S10e, Note 10 9 8,S9 S8, A10e, A20, A50, A70, Google Pixel, LG",False,
...,...,...,...
995,rooCASE Capacitive (Red) Stylus for ASUS Eee Pad Transformer TF101-A1 10.1-Inch,False,
996,"9 Layers Sliced Case Box, Aluminium Heatsink, Wall Charger and HDMI to HDMI Cable for Raspberry Pi Plus/Pi 2 Model B, EEEKit 4 in 1 Starter Kit",False,
997,Dream Wireless Rubber Case for iPhone 4/4S - Retail Packaging - Black,False,
998,OOPKINS Case Compatible with Samsung Galaxy A13 5G Painted PU Leather Durable Wallet Case with Card Slots Kickstand Magnetic Buckle Protection Cover for Samsung Galaxy A13 5G Road Cat Tiger YB,False,


In [42]:
# Separate predicted smartphones and non-smartphones
predicted_phones = validation_review[
    validation_review["candidate"] == True
]

predicted_accessories = validation_review[
    validation_review["candidate"] == False
]

print("Predicted smartphones:", len(predicted_phones))
print("Predicted accessories:", len(predicted_accessories))

# Inspect every predicted smartphone
display(predicted_phones[["title", "candidate"]])

# Randomly inspect 30 predicted accessories
display(
    predicted_accessories.sample(
        n=min(30, len(predicted_accessories)),
        random_state=42
    )[["title", "candidate"]]
)

Predicted smartphones: 11
Predicted accessories: 989


,title,candidate
130,"Apple iPhone 4S 8 GB AT&T, White",True
159,"Samsung Galaxy S21+ 5G, US Version, 128GB, Phantom Violet - Unlocked (Renewed)",True
285,Motorola RAZR V3 Unlocked Phone with Camera and Video Player-U.S. Version with Warranty (Pink),True
497,"Samsung Galaxy A72 (A725M/DS) 128GB Dual SIM, GSM Unlocked, Android Smartphone (Awesome Black) (Renewed)",True
570,TORU CX PRO iPhone 6 - Hot Pink,True
598,JANIZZ New Unlocked Simple Big Button Dual Screen Flip Mobile Cell Phone for Seniors Elderly with Sos Button and Cradle,True
599,"BLU Vivo XL5-6.3"" HD Display Smartphone, 64GB+ 3GB RAM- Black",True
676,"Samsung Galaxy S6 Active G890A Unlocked GSM Smartphone, Blue",True
897,"BLU Studio Mega (8GB) - 6.0"" HD Dual SIM GSM Factory Unlocked Smartphone (Black)",True
981,"Motorola Moto One Macro (64GB, 4GB) 6.2”, Macro Vision Camera (5X Closer) Dual SIM GSM Unlocked (at&T/T-Mobile/MetroPCS/Cricket/H2O) XT2016-2 International Model (Space Blue, 64 GB)",True


,title,candidate
215,"iPhone 6S Case, MIZOO [Rose Gold]-[Wallet Card Slot][Heavy Duty Protection] For Apple iPhone 6 6S 4.7 with FREE [Tempered Glass Screen Protector] (Rose Gold/Black)",False
334,Beyond Cell Slim Dual Layer Wallet Case Compatible with LG G7 ThinQ/ + Wireless Charging Compatible - Leopard Skin Print,False
505,"HTC M9 Case, rooCASE [Stand Feature] HTC One M9 Wallet Case [Prestige Folio] - Premium Soft PU Leather Wallet Case Flip Cover with Stand and Credit Card ID Holders for HTC One M9 (2015) Model - Black",False
312,"iPhone Xs iPhone X Silicone Case, Soft Touch, Comfortable Grip, TIAMAT Liquid Silicone Case with Microfiber Cloth Lining and Wireless Charging Compatible for iPhone X/iPhone Xs (5.8inch) (Rose Gold)",False
88,for Samsung Galaxy Note 8 Back Screen Protector 3D Carbon Fiber Ultra Thin Protective Film-2 Packs-Transparent,False
539,Bessmate™ Kickstand 2 Meters Underwater Waterproof Galaxy Note 4 Case Shockproof Snowproof Dirtpoof Full Protection Case Cover for Samsung Galaxy Note 4 (Blue). Please buy authorized Bessmate brand and crack down on knockoff,False
282,"Galaxy S6 Edge Case, Samsung Galaxy S6 Edge Case, BENTOBEN Slim Soft TPU Hard PC Hybrid Dual Layer Shockproof Rugged Bumper Protective Phone Casea Cover for Samsung Galaxy S6 Edge,Black",False
107,anccer Compatible for Huawei P30 Lite Case [Colorful Series] [Ultra Thin] PC Material Slim Cover for Huawei P30 Lite (Not for Huawei P30&P30 Pro) (Gravel Green),False
59,"Galaxy Note 9 Case,Hybrid Rugged Hard Back Cover with ID Credit Card Slot Holder Wallet Case for Samsung Galaxy Note 9 Blue",False
510,"LUCKYCAT Compatible with iPhone XR Case with Screen Protectors,Crystal Matte Clear Case,Shockproof Protection Scratch-Resisitant TPU Cover for iPhone XR 6.1 Inch (2018)",False


In [30]:

test_titles = [
    "Samsung Galaxy S21+ 5G Smartphone",
    "Apple iPhone 14 Plus",
    "Sony Xperia XZ Unlocked Smartphone",
    "Samsung Galaxy S21 phone case",
    "Cell Phone Signal Booster for Home",
    "MMOBIEL Dock Connector Compatible with Samsung Galaxy",
    "Universal Mobile Phone Kickstand",
    "NEEWER Phone Stabilizer Video Rig Kit",
]

for title in test_titles:
    print(f"{is_candidate_smartphone(title)!s:5} | {title}")


True  | Samsung Galaxy S21+ 5G Smartphone
True  | Apple iPhone 14 Plus
True  | Sony Xperia XZ Unlocked Smartphone
False | Samsung Galaxy S21 phone case
False | Cell Phone Signal Booster for Home
False | MMOBIEL Dock Connector Compatible with Samsung Galaxy
False | Universal Mobile Phone Kickstand
False | NEEWER Phone Stabilizer Video Rig Kit


In [31]:

pd.set_option("display.max_colwidth", None)

predicted_phones = validation_df.loc[
    validation_df["candidate"],
    ["title"]
].reset_index(drop=True)

print("Predicted smartphone titles:", len(predicted_phones))
display(predicted_phones)


Predicted smartphone titles: 19


,title
0,"Compatible with Anti-drop iPhone 11/XR, HD Ceramic Film, Anti-peep frosted Ceramic Film, Anti-fingerprint Ceramic Film, 2PCS"
1,"Apple iPhone 4S 8 GB AT&T, White"
2,"Samsung Galaxy S21+ 5G, US Version, 128GB, Phantom Violet - Unlocked (Renewed)"
3,Motorola RAZR V3 Unlocked Phone with Camera and Video Player-U.S. Version with Warranty (Pink)
4,Tech21 Evo Tactical Extreme Edition for iPhone 7 - Black
5,"Samsung Galaxy A72 (A725M/DS) 128GB Dual SIM, GSM Unlocked, Android Smartphone (Awesome Black) (Renewed)"
6,Cbus Wireless Three LCD Screen Guards / Protectors for Samsung Galaxy Prevail / Precedent / M820
7,"eTECH Collection 5 Pack of Front & Back Combo PET Plastic Anti Glare Matted Screen Protectors for Apple iPhone 4S/4 AT&T, T-Mobile, Sprint, Verizon - from USA"
8,TORU CX PRO iPhone 6 - Hot Pink
9,JANIZZ New Unlocked Simple Big Button Dual Screen Flip Mobile Cell Phone for Seniors Elderly with Sos Button and Cradle


In [43]:

# Manually reviewed labels for the 19 predicted products.
# Labels: smartphone, feature_phone, accessory, uncertain

manual_labels = {
    0: "accessory",
    1: "smartphone",
    2: "smartphone",
    3: "feature_phone",
    4: "accessory",
    5: "smartphone",
    6: "accessory",
    7: "accessory",
    8: "uncertain",
    9: "feature_phone",
    10: "smartphone",
    11: "uncertain",
    12: "smartphone",
    13: "accessory",
    14: "smartphone",
    15: "accessory",
    16: "accessory",
    17: "smartphone",
    18: "smartphone",
}

predicted_phones["manual_label"] = pd.Series(manual_labels)

display(predicted_phones)
print("\nManual label counts:")
print(predicted_phones["manual_label"].value_counts())


/var/folders/bt/bkjfjwgd46v7pkfqyrdnx6jm0000gn/T/ipykernel_11008/1395586701.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  predicted_phones["manual_label"] = pd.Series(manual_labels)


,title,candidate,manual_label
130,"Apple iPhone 4S 8 GB AT&T, White",True,NaN
159,"Samsung Galaxy S21+ 5G, US Version, 128GB, Phantom Violet - Unlocked (Renewed)",True,NaN
285,Motorola RAZR V3 Unlocked Phone with Camera and Video Player-U.S. Version with Warranty (Pink),True,NaN
497,"Samsung Galaxy A72 (A725M/DS) 128GB Dual SIM, GSM Unlocked, Android Smartphone (Awesome Black) (Renewed)",True,NaN
570,TORU CX PRO iPhone 6 - Hot Pink,True,NaN
598,JANIZZ New Unlocked Simple Big Button Dual Screen Flip Mobile Cell Phone for Seniors Elderly with Sos Button and Cradle,True,NaN
599,"BLU Vivo XL5-6.3"" HD Display Smartphone, 64GB+ 3GB RAM- Black",True,NaN
676,"Samsung Galaxy S6 Active G890A Unlocked GSM Smartphone, Blue",True,NaN
897,"BLU Studio Mega (8GB) - 6.0"" HD Dual SIM GSM Factory Unlocked Smartphone (Black)",True,NaN
981,"Motorola Moto One Macro (64GB, 4GB) 6.2”, Macro Vision Camera (5X Closer) Dual SIM GSM Unlocked (at&T/T-Mobile/MetroPCS/Cricket/H2O) XT2016-2 International Model (Space Blue, 64 GB)",True,NaN



Manual label counts:
Series([], Name: count, dtype: int64)


In [33]:

# Review false positives from the manually inspected candidates

false_positives = predicted_phones[
    predicted_phones["manual_label"].isin(["accessory", "uncertain"])
]

print("Products incorrectly or uncertainly selected:")
display(false_positives[["title", "manual_label"]])


Products incorrectly or uncertainly selected:


,title,manual_label
0,"Compatible with Anti-drop iPhone 11/XR, HD Ceramic Film, Anti-peep frosted Ceramic Film, Anti-fingerprint Ceramic Film, 2PCS",accessory
4,Tech21 Evo Tactical Extreme Edition for iPhone 7 - Black,accessory
6,Cbus Wireless Three LCD Screen Guards / Protectors for Samsung Galaxy Prevail / Precedent / M820,accessory
7,"eTECH Collection 5 Pack of Front & Back Combo PET Plastic Anti Glare Matted Screen Protectors for Apple iPhone 4S/4 AT&T, T-Mobile, Sprint, Verizon - from USA",accessory
8,TORU CX PRO iPhone 6 - Hot Pink,uncertain
11,Piel Frama Unipur for Samsung Galaxy S4 (black),uncertain
13,XtremeMac IPU-IA2-13 InCharge Auto for iPhone/iPod/iPad,accessory
15,Pug Dog Cell Phone/Smartphone Charm Accessory,accessory
16,Popular Game of thrones for Apple Iphone 5 U115690,accessory


In [44]:

# Save the current candidate list for manual review

candidate_review = predicted_phones.copy()

candidate_review["manual_label"] = [
    "smartphone",
    "smartphone",
    "feature_phone",
    "smartphone",
    "uncertain",
    "feature_phone",
    "smartphone",
    "smartphone",
    "smartphone",
    "smartphone",
    "uncertain",
]

display(candidate_review)
print("\nManual review summary:")
print(candidate_review["manual_label"].value_counts())


,title,candidate,manual_label
130,"Apple iPhone 4S 8 GB AT&T, White",True,smartphone
159,"Samsung Galaxy S21+ 5G, US Version, 128GB, Phantom Violet - Unlocked (Renewed)",True,smartphone
285,Motorola RAZR V3 Unlocked Phone with Camera and Video Player-U.S. Version with Warranty (Pink),True,feature_phone
497,"Samsung Galaxy A72 (A725M/DS) 128GB Dual SIM, GSM Unlocked, Android Smartphone (Awesome Black) (Renewed)",True,smartphone
570,TORU CX PRO iPhone 6 - Hot Pink,True,uncertain
598,JANIZZ New Unlocked Simple Big Button Dual Screen Flip Mobile Cell Phone for Seniors Elderly with Sos Button and Cradle,True,feature_phone
599,"BLU Vivo XL5-6.3"" HD Display Smartphone, 64GB+ 3GB RAM- Black",True,smartphone
676,"Samsung Galaxy S6 Active G890A Unlocked GSM Smartphone, Blue",True,smartphone
897,"BLU Studio Mega (8GB) - 6.0"" HD Dual SIM GSM Factory Unlocked Smartphone (Black)",True,smartphone
981,"Motorola Moto One Macro (64GB, 4GB) 6.2”, Macro Vision Camera (5X Closer) Dual SIM GSM Unlocked (at&T/T-Mobile/MetroPCS/Cricket/H2O) XT2016-2 International Model (Space Blue, 64 GB)",True,smartphone



Manual review summary:
manual_label
smartphone       7
feature_phone    2
uncertain        2
Name: count, dtype: int64


In [45]:

# Inspect full metadata for the two uncertain products

uncertain_hints = [
    "TORU CX PRO iPhone 6",
    "CELLALLURE Cell Allure Miracle S",
]

uncertain_metadata = []

with gzip.open(metadata_path, "rt", encoding="utf-8") as file:
    for line in file:
        product = json.loads(line)
        title = str(product.get("title", ""))

        if any(hint.lower() in title.lower() for hint in uncertain_hints):
            uncertain_metadata.append({
                "title": title,
                "parent_asin": product.get("parent_asin"),
                "main_category": product.get("main_category"),
                "features": product.get("features"),
                "description": product.get("description"),
                "details": product.get("details"),
            })

display(pd.DataFrame(uncertain_metadata))


,title,parent_asin,main_category,features,description,details
0,"CELLALLURE Cell Allure Miracle S 6"" HD Screen Gold Unlocked Smartphone",B01N533TXR,Cell Phones & Accessories,"[4G Gsm, 6.0"" HD IPS screen, Quad-Core high speed GHz processor, 7.9mm thin, Unlocked for all GSM carriers - international capable]","[Introducing the newest member to the CELLALLURE smartphones family the miracle 6.0 S. The evolution of the miracle lineup is here with a 16GB internal memory. With a magnificent HD 6"" screen is ideal for streaming media, playing games and now for taking selfies with its front camera selfie flash. Available in Black - White - Gold]","{'Product Dimensions': '7.2 x 3.9 x 1.7 inches', 'Item Weight': '4.8 ounces', 'Item model number': 'CAPHG43-03', 'Batteries': '1 Lithium Ion batteries required. (included)', 'Is Discontinued By Manufacturer': 'No', 'OS': 'Android', 'Wireless communication technologies': 'Cellular', 'Connectivity technologies': 'Wi-Fi', 'Display technology': 'LCD', 'Display resolution': '1280 x 720 pixels', 'Other display features': 'Wireless', 'Human Interface Input': 'Touchscreen', 'Other camera features': '8.0', 'Form Factor': 'Touchscreen', 'Color': 'Gold', 'Whats in the box': '1 Data cable, 1 Protective cover, 1 User manual, 1 Handset, 1 Battery, 1 Charger, 1 Screen protector', 'Manufacturer': 'CellAllure', 'Date First Available': 'October 15, 2016', 'Brand': 'CELLALLURE', 'Model Name': 'MIRACLE 6.0 S', 'Wireless Carrier': 'Unlocked', 'Cellular Technology': '4G', 'Memory Storage Capacity': '16 GB', 'Screen Size': '6 Inches', 'Wireless network technology': 'GSM'}"
1,TORU CX PRO iPhone 6 - Red,B00OFMA0V8,Cell Phones & Accessories,[],[],"{'Product Dimensions': '5.59 x 2.76 x 0.51 inches', 'Item Weight': '1.76 ounces', 'Item model number': '116SSEC-RD', 'Is Discontinued By Manufacturer': 'No', 'Other display features': 'Wireless', 'Form Factor': 'Bumper', 'Manufacturer': 'iMODE', 'Date First Available': 'January 27, 2015', 'Brand': 'TORU', 'Compatible Phone Models': 'iPhone 6', 'Pattern': 'Solid', 'Theme': 'No Theme'}"
2,TORU CX PRO iPhone 6 - American Flag,B00T40FVGK,Cell Phones & Accessories,[],[],"{'Product Dimensions': '5.56 x 2.77 x 0.52 inches', 'Item Weight': '0.353 ounces', 'Item model number': '116SSEC-AF', 'Is Discontinued By Manufacturer': 'No', 'Other display features': 'Wireless', 'Manufacturer': 'iMODE', 'Date First Available': 'July 31, 2015', 'Brand': 'TORU', 'Compatible Phone Models': 'iPhone 6', 'Compatible Devices': 'Apple iPhone 6', 'Screen Size': '4.7 Inches', 'Item Dimensions LxWxH': '5.56 x 2.77 x 0.52 inches'}"
3,TORU CX PRO iPhone 6 - Circle Geo,B015GTLXUA,Cell Phones & Accessories,[],[],"{'Product Dimensions': '5.59 x 2.76 x 0.51 inches', 'Item Weight': '0.352 ounces', 'Item model number': '116TCXP-CG', 'Is Discontinued By Manufacturer': 'No', 'Other display features': 'Wireless', 'Form Factor': 'Bumper', 'Manufacturer': 'iMODE', 'Date First Available': 'October 25, 2015', 'Brand': 'TORU', 'Compatible Phone Models': 'IPhone 6', 'Theme': 'No Theme', 'Compatible Devices': 'Apple iPhone 6'}"
4,TORU CX PRO iPhone 6 - Marble Stone,B06Y25FWWD,Cell Phones & Accessories,[],[],"{'Product Dimensions': '5.56 x 2.77 x 0.52 inches', 'Item Weight': '1.65 ounces', 'Item model number': '116TCXP-ML', 'Is Discontinued By Manufacturer': 'No', 'Other display features': 'Wireless', 'Form Factor': 'Bumper', 'Whats in the box': 'Card Holder', 'Manufacturer': 'iMODE', 'Date First Available': 'May 26, 2017', 'Brand': 'TORU', 'Compatible Phone Models': 'iPhone 6', 'Theme': 'No Theme', 'Included Components': 'Card Holder'}"
5,TORU CX PRO iPhone 6 Plus - Hot Pink,B015DX5EJ0,Cell Phones & Accessories,[],[],"{'Product Dimensions': '6.37 x 3.18 x 0.59 inches', 'Item Weight': '0.798 ounces', 'Item model number': '116PTCXP-HP', 'Is Discontinued By Manufacturer': 'No', 'Other display features': 'Wireless', 'Manufacturer': 'iMODE', 'Date First Available': 'November 5, 2015'}"
6,TORU CX PRO iPhone 6 Plus - Gunmetal,B0

In [46]:
candidate_review.loc[
    candidate_review["title"].str.contains(
        "TORU CX PRO iPhone 6", case=False, na=False
    ),
    "manual_label"
] = "accessory"

candidate_review.loc[
    candidate_review["title"].str.contains(
        "CELLALLURE Cell Allure Miracle S", case=False, na=False
    ),
    "manual_label"
] = "smartphone"

display(candidate_review)
print(candidate_review["manual_label"].value_counts())

,title,candidate,manual_label
130,"Apple iPhone 4S 8 GB AT&T, White",True,smartphone
159,"Samsung Galaxy S21+ 5G, US Version, 128GB, Phantom Violet - Unlocked (Renewed)",True,smartphone
285,Motorola RAZR V3 Unlocked Phone with Camera and Video Player-U.S. Version with Warranty (Pink),True,feature_phone
497,"Samsung Galaxy A72 (A725M/DS) 128GB Dual SIM, GSM Unlocked, Android Smartphone (Awesome Black) (Renewed)",True,smartphone
570,TORU CX PRO iPhone 6 - Hot Pink,True,accessory
598,JANIZZ New Unlocked Simple Big Button Dual Screen Flip Mobile Cell Phone for Seniors Elderly with Sos Button and Cradle,True,feature_phone
599,"BLU Vivo XL5-6.3"" HD Display Smartphone, 64GB+ 3GB RAM- Black",True,smartphone
676,"Samsung Galaxy S6 Active G890A Unlocked GSM Smartphone, Blue",True,smartphone
897,"BLU Studio Mega (8GB) - 6.0"" HD Dual SIM GSM Factory Unlocked Smartphone (Black)",True,smartphone
981,"Motorola Moto One Macro (64GB, 4GB) 6.2”, Macro Vision Camera (5X Closer) Dual SIM GSM Unlocked (at&T/T-Mobile/MetroPCS/Cricket/H2O) XT2016-2 International Model (Space Blue, 64 GB)",True,smartphone


manual_label
smartphone       8
feature_phone    2
accessory        1
Name: count, dtype: int64


In [47]:
total_checked = len(candidate_review)
smartphones = (candidate_review["manual_label"] == "smartphone").sum()
precision = smartphones / total_checked

print(f"Candidates manually checked: {total_checked}")
print(f"Confirmed smartphones: {smartphones}")
print(f"Preliminary smartphone precision: {precision:.2%}")

Candidates manually checked: 11
Confirmed smartphones: 8
Preliminary smartphone precision: 72.73%


### Validation Summary

A preliminary validation of 11 candidate products identified 8 smartphones, 2 feature phones, and 1 accessory. The smartphone filter achieved a preliminary precision of 72.73% on this manually reviewed candidate set.

The results show that title-based keyword filtering can identify smartphone products but may also include feature phones and accessories. The filter therefore requires further refinement and independent validation before being used to construct the final dataset.

The validation sample is small, and the filtering rules were refined using the same sample. Consequently, these results should not be interpreted as an unbiased estimate of performance on unseen products.